# 1 — Estágio A: ensinar PORTUGUÊS ao CSM-1B · Colab

**Por quê:** o teste 0.x (local, M2) provou que a VOZ entra nos pesos com 48min
(spk-sim 0.97+ sem âncora), mas a LÍNGUA não (WER 100%, undertraining). Este
notebook ensina o português; o **notebook 2** põe a sua voz por cima (Estágio B).

**Receita-referência (georgiano, único caso com métrica):** LoRA r=64/α=64, LR 5e-5
cosine, ~14 épocas, batch-eff ~128, dados LIMPOS → CER 2,8%.

**Experimentos** (matriz em `specs/EXPERIMENTS.md`): `A1_cml` (68h CC-BY, **COMECE
AQUI**) · `A3_tagarela` (🔥 podcast espontâneo, aposta) · `A2_mix` (~416h leitura).
Sempre **SMOKE=True** (5h/1 época) antes do run cheio — ouvir antes de gastar.

**GPU:** A100 ideal; o batch se auto-ajusta (T4 grátis também roda o smoke, mais
devagar). Checkpoints no Drive — sessão pode cair que retoma sozinho.

In [ ]:
# ⚙️ ESCOLHA DO EXPERIMENTO  (matriz + hipóteses: specs/EXPERIMENTS.md)
# Política 2026-06-11: fase RESEARCH — qualquer dataset entra; linhagem no nome do run.
EXPERIMENT = 'A1_cml'        # 'A1_cml' | 'A3_tagarela' | 'A2_mix'
SMOKE = True                 # True = 5h/1 época (ouvir antes de gastar); False = run cheio
EPOCHS = 1 if SMOKE else 8   # A1 cheio: 6-10 épocas; A3 com 100-300h: 2-4 épocas
MAX_HOURS_DATA = 5 if SMOKE else None    # A3 cheio: setar 100-300 (slice do stream)
LORA_R, LORA_ALPHA, LR = 64, 64, 5e-5
RUN = f"csm_{EXPERIMENT}{'_smoke' if SMOKE else ''}"

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
GH_TOKEN = userdata.get('GH_TOKEN'); os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!git clone https://{GH_TOKEN}@github.com/pedrocormann/TTS-ptbr.git /content/TTS-ptbr 2>/dev/null || (cd /content/TTS-ptbr && git pull)
%cd /content/TTS-ptbr
DRIVE = '/content/drive/MyDrive/TTS-ptbr-data'
!mkdir -p {DRIVE}/runs

In [ ]:
%%capture
# transformers 4.52.3 = versão EXATA do checkpoint csm-1b (transformers 5.x renomeia
# pesos → "embed_audio_tokens MISSING"). torchao do Colab (0.10) é rejeitado pelo peft
# (quer >0.16) e não usamos → remover. HF puro, sem unsloth.
!pip install "transformers==4.52.3" peft accelerate "datasets>=3.4.1,<4.0.0" \
    soundfile jiwer librosa soxr bitsandbytes torchcodec faster-whisper==1.1.0
!pip uninstall -y torchao

## 1. Dados por experimento (24 kHz mono · texto cru · speaker id por falante)

In [ ]:
from datasets import load_dataset, Audio
import warnings

def cap_hours(ds, hours, dur_key=None):
    if hours is None: return ds
    total, idx = 0.0, []
    for i, ex in enumerate(ds):
        a = ex['audio']; total += len(a['array'])/a['sampling_rate']
        idx.append(i)
        if total >= hours*3600: break
    return ds.select(idx)

if EXPERIMENT == 'A1_cml':
    # CML-TTS pt (Frederico/Oliveira et al., CC-BY 4.0): 68h, 30 falantes, LIMPO
    raw = load_dataset('ylacombe/cml-tts', 'portuguese', split='train')
    TEXT_COL = 'text'
elif EXPERIMENT == 'A2_mix':
    from datasets import concatenate_datasets
    cml = load_dataset('ylacombe/cml-tts', 'portuguese', split='train')
    mls = load_dataset('facebook/multilingual_librispeech', 'portuguese', split='train')
    parts = [cml, mls]
    # Common Voice pt: aceite os termos no HF antes; se falhar, segue sem
    try:
        cv = load_dataset('mozilla-foundation/common_voice_17_0', 'pt', split='train', trust_remote_code=True)
        parts.append(cv)
    except Exception as e:
        warnings.warn(f'CV pulado: {e}')
    norm = []
    for p in parts:
        cols = p.column_names
        tcol = 'text' if 'text' in cols else ('transcript' if 'transcript' in cols else 'sentence')
        p = p.rename_column(tcol, 'text') if tcol != 'text' else p
        norm.append(p.remove_columns([c for c in p.column_names if c not in ('audio','text')]))
    raw = concatenate_datasets(norm).shuffle(seed=42)
    TEXT_COL = 'text'
elif EXPERIMENT == 'A3_tagarela':
    # TAGARELA (Frederico, 8.972h de podcast ESPONTÂNEO; linhagem NC — anotada
    # no nome do run; retreino limpo documentado p/ produto). Hipótese: fala
    # espontânea ensina pt conversacional melhor que leitura. Stream → slice.
    print(f'🔥 A3-tagarela: slice de {MAX_HOURS_DATA or 100}h via streaming (1,2TB não baixa)')
    stream = load_dataset('freds0/TAGARELA', split='train', streaming=True)
    rows = []
    total, target = 0.0, (MAX_HOURS_DATA or 100)*3600
    for ex in stream:
        rows.append({'audio': ex['audio'], 'text': ex['sentence']})
        total += len(ex['audio']['array'])/ex['audio']['sampling_rate']
        if total >= target: break
    from datasets import Dataset
    raw = Dataset.from_list(rows)
    TEXT_COL = 'text'

raw = raw.cast_column('audio', Audio(sampling_rate=24000))
if EXPERIMENT != 'A3_tagarela':
    raw = cap_hours(raw.shuffle(seed=42), MAX_HOURS_DATA)
print(raw)

In [ ]:
# filtros de higiene (lições: limpo > horas; nada de clipe gigante/vazio)
import numpy as np
def ok(ex):
    a = ex['audio']['array']; d = len(a)/24000
    return 1.5 <= d <= 20.0 and len(str(ex[TEXT_COL]).split()) >= 3
raw = raw.filter(ok)
MAX_AUDIO = int(max(len(ex['audio']['array']) for ex in raw.select(range(min(2000, len(raw)))))) + 1
MAX_AUDIO = min(MAX_AUDIO, 20*24000+1)
total_h = sum(len(ex['audio']['array']) for ex in raw.select(range(min(2000,len(raw)))))/24000/3600 * len(raw)/min(2000,len(raw))
print(f'{len(raw)} clipes · ~{total_h:.0f}h estimadas · max_audio={MAX_AUDIO/24000:.0f}s')

## 2. Modelo + preprocess (multi-speaker: 1 id por falante ajuda o modelo a separar voz de língua)

In [ ]:
from transformers import AutoProcessor, CsmForConditionalGeneration
import torch, hashlib

MODEL_ID = 'unsloth/csm-1b'        # mirror Apache ungated (mesmos pesos do sesame/csm-1b)
BF16 = torch.cuda.is_bf16_supported()
processor = AutoProcessor.from_pretrained(MODEL_ID)
model, _info = CsmForConditionalGeneration.from_pretrained(
    MODEL_ID, output_loading_info=True)  # float32: o codec Mimi gera float32, então o modelo
    # NÃO pode ser bf16 (mismatch no merge). bf16=True no Trainer faz autocast.
_crit = [k for k in _info.get('missing_keys', []) if 'embed_audio' in k]
assert not _crit, f'❌ pesos de ÁUDIO faltando ({_crit}) — transformers incompatível com o checkpoint'
model = model.to('cuda')
model.train(); model.codec_model.eval()   # codec Mimi congelado (exemplo oficial HF)

def spk_id(ex, i):                 # 1 id por falante ajuda a separar voz de língua
    return str(int(hashlib.md5(str(ex.get('speaker_id', i)).encode()).hexdigest(), 16) % 10)

def preprocess(ex, idx):
    conv = [{'role': spk_id(ex, idx), 'content': [
        {'type': 'text', 'text': str(ex[TEXT_COL]).strip()},
        {'type': 'audio', 'path': ex['audio']['array']}]}]
    out = processor.apply_chat_template(
        conv, tokenize=True, return_dict=True, output_labels=True,
        text_kwargs={'padding': 'max_length', 'max_length': 256,
                     'pad_to_multiple_of': 8, 'padding_side': 'right'},
        audio_kwargs={'sampling_rate': 24000, 'max_length': MAX_AUDIO, 'padding': 'max_length'},
        common_kwargs={'return_tensors': 'pt'})
    return {k: v[0] for k, v in out.items()}

ds = raw.map(preprocess, with_indices=True, remove_columns=raw.column_names, desc='tokenizando')
print(ds)

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer
import glob

cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, cfg); model.print_trainable_parameters()

class CSMTrainer(Trainer):   # mantém o codec Mimi em eval (não treina)
    def training_step(self, model, inputs, *args, **kwargs):
        bm = model.get_base_model() if hasattr(model, 'get_base_model') else model
        if hasattr(bm, 'codec_model'): bm.codec_model.eval()
        return super().training_step(model, inputs, *args, **kwargs)

BATCH, ACCUM = 2, 16               # efetivo 32 (conservador; sobe depois que validar)
OUT = f'{DRIVE}/runs/{RUN}'
resume = bool(glob.glob(f'{OUT}/checkpoint-*'))

trainer = CSMTrainer(
    model=model, train_dataset=ds,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
        num_train_epochs=EPOCHS, learning_rate=LR,
        lr_scheduler_type='cosine', warmup_ratio=0.03,
        bf16=BF16, fp16=not BF16,
        logging_steps=10, optim='adamw_8bit', weight_decay=0.01, seed=3407,
        output_dir=OUT, report_to='none', save_steps=200, save_total_limit=3,
        remove_unused_columns=False))
trainer.train(resume_from_checkpoint=resume)
model.save_pretrained(f'{OUT}/final'); processor.save_pretrained(f'{OUT}/final')
print('✅ BASE-PT salva em', f'{OUT}/final')

## 3. Eval do gate (WER + sondas de pronúncia + amostras pra ouvir)

In [ ]:
import json, pathlib, soundfile as sf
model.eval()
ref = raw[0]  # âncora de voz qualquer do corpus (Estágio A não é sobre voz)
bench = [json.loads(l) for l in open('eval/benchmark_ptbr.jsonl', encoding='utf-8') if l.strip()]
probes = [json.loads(l) for l in open('eval/benchmark_pronuncia_ptbr.jsonl', encoding='utf-8') if l.strip()][:8]
outdir = pathlib.Path(f'gen_{RUN}'); outdir.mkdir(exist_ok=True)
for i, item in enumerate(bench + probes):
    conv = [{'role': '0', 'content': [{'type': 'text', 'text': str(ref[TEXT_COL])},
                                       {'type': 'audio', 'path': ref['audio']['array']}]},
            {'role': '0', 'content': [{'type': 'text', 'text': item['text']}]}]
    inputs = processor.apply_chat_template(conv, tokenize=True, return_dict=True)
    with torch.no_grad():
        audio = model.generate(**inputs.to('cuda'), max_new_tokens=375, output_audio=True,
                               do_sample=True, temperature=0.9, depth_decoder_do_sample=True,
                               depth_decoder_temperature=0.9)
    sf.write(outdir / f"{item.get('id', i)}.wav", audio[0].to(torch.float32).cpu().numpy(), 24000)
!pip -q install faster-whisper==1.1.0
!python -m eval.wer_roundtrip --in-dir gen_{RUN} --transcripts eval/benchmark_ptbr.jsonl --model medium --lang pt
!cp -r gen_{RUN} {DRIVE}/runs/{RUN}/   # amostras no Drive pra escuta
print('GATE: WER<=15% + sondas>=70% de ouvido + soa pt-BR → registrar em specs/EXPERIMENTS.md')

## 4. Próximo: Estágio B (voz do Pedro sobre a BASE-PT)
No notebook 2 (2_csm_voz_pedro), troque `model_name='unsloth/csm-1b'` por
`model_name=f'{DRIVE}/runs/<RUN_VENCEDOR>/final'` (a BASE-PT) e treine a voz por cima.
Expectativa (B1 da matriz): WER≤15% **E** spk-sim≥0.95 sem âncora — voz+língua juntas.

📒 Linhagem: registre o resultado em `specs/EXPERIMENTS.md` (run, dados, métricas).
Pesos com dado NC (A3) experimentam à vontade; pro PRODUTO, retreino com linhagem
limpa (A1/A2/dados próprios) — caminho já documentado na matriz.